# VIR Vbeta - G2 Throughput (Burn CUDA vs PyTorch)
Ukur tok/s forward VQ di GPU Kaggle. Tulis log.txt + result.json.
Catatan: bandingkan apples-to-apples (matmul + argmin, fp32, sync).

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

In [ ]:
import subprocess, json, time, os
log = open('/kaggle/working/log.txt','w')
def P(*a):
    s=' '.join(str(x) for x in a)
    print(s); log.write(s+'\n'); log.flush()
t0=time.time()
r=subprocess.run(['git','clone','--depth','1','https://github.com/Gomobra/VIR.git','/tmp/VIR'], capture_output=True, text=True)
P('clone rc', r.returncode)
env={**os.environ,'CARGO_TARGET_DIR':'/tmp/target'}
b=subprocess.run(['cargo','build','--release','--manifest-path','/tmp/VIR/crates/vir-vq/Cargo.toml','--features','cuda','--bin','bench'], capture_output=True, text=True, env=env)
P('build rc', b.returncode); P(b.stderr[-3000:])
r=subprocess.run(['/tmp/target/release/bench'], capture_output=True, text=True, env=env)
P('BURN:', r.stdout.strip()); P(r.stderr[-1000:])
open('/kaggle/working/result.json','w').write(json.dumps({'burn': r.stdout.strip(), 'elapsed': time.time()-t0}))
log.close()

In [ ]:
import torch, time
log = open('/kaggle/working/log.txt','a')
dev='cuda'
V,D=4096,384
E=torch.randn(V,D,device=dev)
z=torch.randn(256*128,D,device=dev)
for _ in range(5):
    cross = z @ E.t()
    idx = cross.argmax(1)
torch.cuda.synchronize()
t0=time.time(); iters=50
for _ in range(iters):
    cross = z @ E.t()
    idx = cross.argmax(1)
torch.cuda.synchronize()
el=time.time()-t0
s=f'PYTORCH: vq_forward toks/s = {256*128*iters/el:.2f} (elapsed {el:.3f}s)'
print(s); log.write(s+'\n'); log.close()